# 04 - Train Metabolic Simulation LSTM

This notebook generates synthetic glucose-insulin dynamics using the Bergman Minimal Model and trains a stacked LSTM to predict glucose 1 hour ahead.

**Targets**

- MAE < 12 mg/dL
- RMSE < 18 mg/dL
- R2 > 0.92

**Outputs**

- `data/metabolic_profiles.pkl`
- `data/metabolic_train_test.npz`
- `data/metabolic_stats.json`
- `models/metabolic_lstm.h5`
- `models/metabolic_lstm_model_info.json`

## Bergman Minimal Model

`dG/dt = -SG * (G - Gb) - X * G + D(t)`

`dX/dt = -p2 * X + p3 * (I - Ib)`

`dI/dt = -n * (I - Ib) + gamma * max(0, G - Gb)`

Where:

- `G`: blood glucose in mg/dL
- `X`: insulin action
- `I`: plasma insulin in microU/mL
- `D(t)`: meal or exercise disturbance
- `SI`, `SG`, `p2`, `p3`, `n`, `gamma`: patient-specific parameters

In [ ]:
# Colab setup if needed:
# !pip install tensorflow scipy pandas numpy

from backend.ml.metabolic_lstm import (
    MetabolicConfig,
    prepare_metabolic_dataset,
    train_metabolic_lstm,
    load_train_test_npz,
    build_metabolic_lstm,
)

# Use patients=100 or 1000 for a quick smoke test.
# Use patients=10000 for the full Colab experiment.
config = MetabolicConfig(
    patients=10000,
    samples_per_patient=64,
    epochs=50,
    batch_size=64,
    dataset_output='data/metabolic_profiles.pkl',
    train_npz_output='data/metabolic_train_test.npz',
    stats_output='data/metabolic_stats.json',
    model_output='models/metabolic_lstm.h5',
    info_output='models/metabolic_lstm_model_info.json',
)
config

## Generate synthetic dataset

Each patient receives randomized insulin sensitivity (`SI`) and glucose effectiveness (`SG`), three meals at 8h/13h/19h, and optional exercise at 10h or 17h. The model samples 120-minute history windows and predicts glucose 60 minutes ahead.

In [ ]:
stats = prepare_metabolic_dataset(config)
stats

## Inspect model architecture

In [ ]:
model = build_metabolic_lstm()
model.summary()

## Train stacked LSTM

Architecture:

- Time-series input: `(120, 2)` for glucose and insulin history
- LSTM(128) -> Dropout(0.3)
- LSTM(64) -> Dropout(0.3)
- LSTM(32)
- Metadata input: time since meal and exercise flag
- Fusion dense layers -> glucose output

In [ ]:
model_info = train_metabolic_lstm(config)
model_info

## Report metrics

In [ ]:
print(f"MAE: {model_info['mae_mg_dl']:.2f} mg/dL (Target: <12 mg/dL)")
print(f"RMSE: {model_info['rmse_mg_dl']:.2f} mg/dL (Target: <18 mg/dL)")
print(f"R2: {model_info['r2_score']:.4f} (Target: >0.92)")
print(f"Target met: {model_info['target_met']}")